# 6. Market-making quotes: Avellaneda-Stoikov

**Level:** L3. **Tags:** `market-making`, `inventory`, `microstructure`.
**Library:** `qp_exec.mm` (`python/execution`).

**Plain idea.** A market maker posts a bid and an ask. Holding inventory is
risky, so a long maker shades both quotes down (to sell more and buy less)
and a short maker shades them up. The spread pays for that risk and for the
chance that a quote is not hit at all.

**Formal statement** (Avellaneda and Stoikov, "High-frequency trading in a
limit order book", *Quantitative Finance* 8(3), 2008, eqs. 29-30): with mid
$s$, inventory $q$, risk aversion $\gamma$, volatility $\sigma$, order
arrival decaying as $A e^{-k\delta}$ and time left $\tau$, the reservation
price is $r = s - q\gamma\sigma^2\tau$ and the total spread is
$\gamma\sigma^2\tau + \frac{2}{\gamma}\ln(1 + \gamma/k)$, centred on $r$.

In [ ]:
import _paths  # noqa: F401  (puts the library on sys.path)

In [ ]:
import math

from qp_exec.mm.quotes import ASParams, avellaneda_stoikov  # noqa: F401

## Task 1: quotes for flat, long and short inventory

With $\gamma = 0.1$, $\sigma = 2$, $k = 1.5$, mid 100 and $\tau = 1$, store
the quotes for $q = -5, 0, 5$ in a dict `quotes` keyed by $q$.

In [ ]:
p = ASParams(gamma=0.1, sigma=2.0, k=1.5)
quotes = ...
# Exercise: replace the ... above, then run the check cell below.

**Independent check.** The two formulas above, written out by hand.

In [ ]:
g, s, k, tau = 0.1, 2.0, 1.5, 1.0
spread = g * s**2 * tau + 2 / g * math.log(1 + g / k)
for q, x in quotes.items():
    r = 100.0 - q * g * s**2 * tau
    assert abs(x.reservation - r) < 1e-12
    assert abs(x.bid - (r - spread / 2)) < 1e-12 and abs(x.ask - (r + spread / 2)) < 1e-12
assert quotes[5].ask < quotes[0].ask < quotes[-5].ask

## Task 2: the spread at the close

As $\tau \to 0$ the inventory term vanishes and the spread falls to
$\frac{2}{\gamma}\ln(1+\gamma/k)$, which tends to $2/k$ as $\gamma \to 0$.
Store the spread at $\tau = 0$ for $\gamma = 10^{-6}$ (inventory 3) as
`close_spread`.

In [ ]:
close_spread = ...
# Exercise: replace the ... above, then run the check cell below.

In [ ]:
assert abs(close_spread - 2 / 1.5) < 1e-5

**Takeaways.** Inventory moves the centre of the quotes, not their width;
time left widens them. `qp_exec.mm.sim.run_mm` runs these quotes against a
simulated order flow, and `qp_exec.mm.quotes.GLFT` adds the inventory bound
of Gueant, Lehalle and Fernandez-Tapia (2013).